# 예제 franka_ex08: FR3 경로 제약조건 — Orientation Constraint

Franka FR3 끝단의 **방향(아래)** 을 *경로 내내* 유지하면서 여러 목표 지점을 순회하는 예제.
"물컵 운반" 시나리오: 가는 도중에도 손목이 뒤집히면 안 된다.

**6-DOF 예제와 다른 점**
- 끝단 링크: `fr3_hand_tcp`, planning group: `fr3_arm`
- 워크스페이스: FR3 reach 에 맞춰 시작점 `(0.45, 0.00, 0.55)`, 좌우 ±15cm 폭
- 7-DOF redundancy 덕에 방향 제약 하에서도 6-DOF 보다 IK 해를 더 쉽게 찾는다 → 같은 제약/타깃 조합에서 성공률↑
- `home` 자세가 SRDF 에 없으므로 시작/복귀는 `ready`
- `use_sim_time=True`

**학습 내용**
- `Constraints.path_constraints` 와 `Constraints.goal_constraints` 의 차이
- `OrientationConstraint` 의 X/Y/Z tolerance 의미
- 제약 하 플래닝 시간 / 시도 횟수 늘려야 하는 이유
- RViz `MarkerArray` 로 시작점, 타깃, 허용 오차 링 시각화

## 실행 절차

이 노트북은 별도로 띄운 MoveIt + RViz 의 `move_group` 액션 서버에 클라이언트로 붙는다.

> ⚠ 다른 로봇용 MoveIt launch 가 떠 있으면 같은 토픽으로 충돌할 수 있다.
> 시작 전에 `pgrep -af 'ros2 launch'` 로 잔존 프로세스가 없는지 확인하자.

### 터미널 1 — Franka FR3 (Gazebo Sim) + MoveIt + RViz 기동

```bash
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
ros2 launch franka_tutorials franka_gazebo_moveit.launch.py
```

RViz 가 뜨면 **`MarkerArray` Display 를 추가하고 Topic 을 `/constraint_markers` 로 설정**한다.
Fixed Frame 은 `fr3_link0` 로 둔다.

### 터미널 2 — Jupyter 기동

```bash
source ~/venv/ros_jazzy/bin/activate
source /opt/ros/jazzy/setup.bash
source ~/robot_arm/install/setup.bash
cd ~/robot_arm/src/robotarm_tutorials/robot_arm_tutorials/robot_arm_tutorials
jupyter lab
```

셀을 위에서 아래로 순서대로 실행한다 (`Shift+Enter`).

## 1. 상수

In [ ]:
PLANNING_GROUP    = 'fr3_arm'
REFERENCE_FRAME   = 'fr3_link0'
END_EFFECTOR_LINK = 'fr3_hand_tcp'
ARM_JOINTS        = ['fr3_joint1', 'fr3_joint2', 'fr3_joint3',
                     'fr3_joint4', 'fr3_joint5', 'fr3_joint6',
                     'fr3_joint7']
MARKER_TOPIC      = '/constraint_markers'

## 2. ROS 2 초기화 + 노드

In [ ]:
import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from rclpy.parameter import Parameter
from sensor_msgs.msg import JointState
from moveit_msgs.action import MoveGroup, ExecuteTrajectory
from visualization_msgs.msg import MarkerArray, Marker
from std_msgs.msg import ColorRGBA

In [ ]:
try:
    rclpy.init()
except RuntimeError:
    pass  # 이미 초기화된 경우 무시

In [ ]:
node = Node(
    'franka_ex08_constraints_demo',
    parameter_overrides=[Parameter('use_sim_time', value=True)],
)
move_client = ActionClient(node, MoveGroup, 'move_action')

joint_state = {'msg': None}
node.create_subscription(
    JointState, 'joint_states',
    lambda msg: joint_state.update(msg=msg), 10,
)
node.get_logger().info('=== franka_ex08 노트북 노드 생성 완료 ===')
marker_pub = node.create_publisher(MarkerArray, MARKER_TOPIC, 10)

## 3. 서버 / `joint_states` 준비

In [ ]:
import time

def wait_for_ready(timeout_sec: float = 30.0) -> None:
    if not move_client.wait_for_server(timeout_sec=timeout_sec):
        raise RuntimeError('MoveGroup 액션 서버 연결 실패')
    start = time.time()
    while joint_state['msg'] is None:
        rclpy.spin_once(node, timeout_sec=0.1)
        if time.time() - start > timeout_sec:
            raise RuntimeError('joint_states 수신 실패')
    node.get_logger().info('action server + /joint_states 준비됨')

wait_for_ready()

## 4. SRDF `ready` 자세

In [ ]:
from rclpy.parameter_client import AsyncParameterClient
import xml.etree.ElementTree as ET

def fetch_srdf_xml(timeout_sec: float = 10.0) -> str:
    client = AsyncParameterClient(node, 'move_group')
    if not client.wait_for_services(timeout_sec=timeout_sec):
        raise RuntimeError('move_group 파라미터 서비스 연결 실패')
    future = client.get_parameters(['robot_description_semantic'])
    rclpy.spin_until_future_complete(node, future, timeout_sec=timeout_sec)
    return future.result().values[0].string_value

def parse_named_pose(srdf_xml: str, name: str, group: str) -> dict:
    root = ET.fromstring(srdf_xml)
    for gs in root.findall('group_state'):
        if gs.attrib.get('group') == group and gs.attrib.get('name') == name:
            return {j.attrib['name']: float(j.attrib.get('value', '0'))
                    for j in gs.findall('joint')}
    raise RuntimeError(f'SRDF group_state "{name}" (group={group}) 없음')

def load_named_pose(name: str, timeout_sec: float = 10.0) -> dict:
    return parse_named_pose(fetch_srdf_xml(timeout_sec), name, PLANNING_GROUP)

ready_target = load_named_pose('ready')
node.get_logger().info(f'ready: {ready_target}')

## 5. Pose / MoveGroup / FK 헬퍼

In [ ]:
import math
import tf_transformations
from geometry_msgs.msg import Pose, Point, Quaternion

def euler_to_quaternion(roll: float, pitch: float, yaw: float) -> Quaternion:
    q = tf_transformations.quaternion_from_euler(roll, pitch, yaw)
    return Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])

def make_pose(x: float, y: float, z: float,
              roll: float = 0.0, pitch: float = 0.0, yaw: float = 0.0) -> Pose:
    pose = Pose()
    pose.position = Point(x=x, y=y, z=z)
    pose.orientation = euler_to_quaternion(roll, pitch, yaw)
    return pose

In [ ]:
from moveit_msgs.msg import (
    Constraints, JointConstraint,
    PositionConstraint, OrientationConstraint, BoundingVolume,
    MotionPlanRequest, PlanningOptions, MoveItErrorCodes,
)
from shape_msgs.msg import SolidPrimitive
from geometry_msgs.msg import Vector3

def make_joint_constraints(joint_values: dict, tol: float = 0.01) -> Constraints:
    c = Constraints()
    for jname, val in joint_values.items():
        c.joint_constraints.append(JointConstraint(
            joint_name=jname, position=val,
            tolerance_above=tol, tolerance_below=tol, weight=1.0,
        ))
    return c

def make_position_constraint(pose: Pose, tol: float = 0.01) -> PositionConstraint:
    pc = PositionConstraint()
    pc.header.frame_id = REFERENCE_FRAME
    pc.link_name = END_EFFECTOR_LINK
    pc.target_point_offset = Vector3(x=0.0, y=0.0, z=0.0)
    bv = BoundingVolume()
    sphere = SolidPrimitive()
    sphere.type = SolidPrimitive.SPHERE
    sphere.dimensions = [tol]
    bv.primitives.append(sphere)
    sp = Pose()
    sp.position = Point(x=pose.position.x, y=pose.position.y, z=pose.position.z)
    sp.orientation.w = 1.0
    bv.primitive_poses.append(sp)
    pc.constraint_region = bv
    pc.weight = 1.0
    return pc

def make_orientation_constraint(pose_or_quat, tol: float = 0.01) -> OrientationConstraint:
    oc = OrientationConstraint()
    oc.header.frame_id = REFERENCE_FRAME
    oc.link_name = END_EFFECTOR_LINK
    if hasattr(pose_or_quat, 'orientation'):
        oc.orientation = pose_or_quat.orientation
    else:
        oc.orientation = pose_or_quat
    oc.absolute_x_axis_tolerance = tol
    oc.absolute_y_axis_tolerance = tol
    oc.absolute_z_axis_tolerance = tol
    oc.weight = 1.0
    return oc

def make_plan_request(vel: float = 0.3, acc: float = 0.3,
                      attempts: int = 5, plan_time: float = 10.0,
                      planner_id: str = '') -> MotionPlanRequest:
    req = MotionPlanRequest()
    req.group_name = PLANNING_GROUP
    req.num_planning_attempts = attempts
    req.allowed_planning_time = plan_time
    req.max_velocity_scaling_factor = vel
    req.max_acceleration_scaling_factor = acc
    if planner_id:
        req.planner_id = planner_id
    return req

def send_move_goal(req: MotionPlanRequest, plan_only: bool = False):
    goal = MoveGroup.Goal()
    goal.request = req
    goal.planning_options = PlanningOptions(
        plan_only=plan_only, replan=not plan_only, replan_attempts=3 if not plan_only else 0)
    sf = move_client.send_goal_async(goal)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return MoveItErrorCodes.PLANNING_FAILED, None
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    res = rf.result().result
    return res.error_code.val, res.planned_trajectory

def go_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'joint goal 실패 error_code={code_val}')
    return ok

def go_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3) -> bool:
    req = make_plan_request(vel, acc)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'pose goal 실패 error_code={code_val} (IK 해 없음 가능)')
    return ok

def plan_to_joint_goal(joint_values: dict, vel: float = 0.3, acc: float = 0.3,
                       planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    req.goal_constraints.append(make_joint_constraints(joint_values))
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

def plan_to_pose_goal(pose: Pose, vel: float = 0.3, acc: float = 0.3,
                      planner_id: str = '', plan_time: float = 10.0):
    req = make_plan_request(vel, acc, plan_time=plan_time, planner_id=planner_id)
    c = Constraints()
    c.position_constraints.append(make_position_constraint(pose))
    c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(c)
    code_val, traj = send_move_goal(req, plan_only=True)
    return code_val == MoveItErrorCodes.SUCCESS, traj

In [ ]:
from moveit_msgs.action import ExecuteTrajectory

execute_client = ActionClient(node, ExecuteTrajectory, 'execute_trajectory')
if not execute_client.wait_for_server(timeout_sec=15.0):
    raise RuntimeError('ExecuteTrajectory 액션 서버 연결 실패')

def execute_trajectory(trajectory) -> bool:
    g = ExecuteTrajectory.Goal()
    g.trajectory = trajectory
    sf = execute_client.send_goal_async(g)
    rclpy.spin_until_future_complete(node, sf)
    handle = sf.result()
    if handle is None or not handle.accepted:
        return False
    rf = handle.get_result_async()
    rclpy.spin_until_future_complete(node, rf)
    return rf.result().result.error_code.val == MoveItErrorCodes.SUCCESS

In [ ]:
from moveit_msgs.srv import GetPositionFK
from moveit_msgs.msg import RobotState

fk_client = node.create_client(GetPositionFK, 'compute_fk')
fk_client.wait_for_service(timeout_sec=10.0)

def trajectory_to_ee_path(trajectory, max_points: int = 60):
    '''RobotTrajectory → fr3_hand_tcp 의 base 기준 좌표 리스트.'''
    jt = trajectory.joint_trajectory
    total = len(jt.points)
    if total == 0:
        return []
    step = max(1, total // max_points)
    indices = list(range(0, total, step))
    if indices[-1] != total - 1:
        indices.append(total - 1)
    pts = []
    for idx in indices:
        req = GetPositionFK.Request()
        req.header.frame_id = REFERENCE_FRAME
        req.fk_link_names = [END_EFFECTOR_LINK]
        rs = RobotState()
        rs.joint_state.name = list(jt.joint_names)
        rs.joint_state.position = list(jt.points[idx].positions)
        req.robot_state = rs
        fut = fk_client.call_async(req)
        rclpy.spin_until_future_complete(node, fut)
        resp = fut.result()
        if resp and resp.error_code.val == MoveItErrorCodes.SUCCESS and resp.pose_stamped:
            p = resp.pose_stamped[0].pose.position
            pts.append((p.x, p.y, p.z))
    return pts

## 6. 경로 제약 — `OrientationConstraint`

`Constraints.path_constraints` 에 넣은 제약은 **경로 전체** 에서 만족되어야 한다.
`absolute_x/y/z_axis_tolerance` 는 라디안 기준 X/Y/Z 회전 허용량.
Z 축 (yaw) 자유도를 풀고 싶으면 큰 값(예: π) 을 주면 된다.

In [ ]:
def go_to_pose_with_path_constraint(pose: Pose, path_orientation: Quaternion,
                                     tol_xy: float = 0.3, tol_z: float = 3.14,
                                     vel: float = 0.2, attempts: int = 15,
                                     plan_time: float = 30.0) -> bool:
    req = make_plan_request(vel=vel, acc=vel, attempts=attempts, plan_time=plan_time)

    # path constraint: 끝단 방향 path_orientation 유지
    path_oc = OrientationConstraint()
    path_oc.header.frame_id = REFERENCE_FRAME
    path_oc.link_name = END_EFFECTOR_LINK
    path_oc.orientation = path_orientation
    path_oc.absolute_x_axis_tolerance = tol_xy
    path_oc.absolute_y_axis_tolerance = tol_xy
    path_oc.absolute_z_axis_tolerance = tol_z
    path_oc.weight = 1.0
    path_constraints = Constraints()
    path_constraints.orientation_constraints.append(path_oc)
    req.path_constraints = path_constraints

    # goal constraint: 위치 + 방향
    goal_c = Constraints()
    goal_c.position_constraints.append(make_position_constraint(pose))
    goal_c.orientation_constraints.append(make_orientation_constraint(pose))
    req.goal_constraints.append(goal_c)

    code_val, _ = send_move_goal(req, plan_only=False)
    ok = (code_val == MoveItErrorCodes.SUCCESS)
    if not ok:
        node.get_logger().error(f'제약 plan/exec 실패 error_code={code_val}')
    return ok

## 7. 마커 — 시작점, 타깃, 허용 오차 링

In [ ]:
from geometry_msgs.msg import Point, Vector3

COLOR_TEXT = ColorRGBA(r=1.0, g=1.0, b=1.0, a=1.0)
COLORS = [
    ColorRGBA(r=0.9, g=0.2, b=0.2, a=1.0),  # A 빨강
    ColorRGBA(r=0.2, g=0.8, b=0.2, a=1.0),  # B 초록
    ColorRGBA(r=0.2, g=0.3, b=0.9, a=1.0),  # C 파랑
]
_markers = MarkerArray()

def publish_targets(start_pos, targets, tol_rad: float):
    stamp = node.get_clock().now().to_msg()
    new_markers = []
    sx, sy, sz = start_pos
    s_sphere = Marker()
    s_sphere.header.frame_id = REFERENCE_FRAME
    s_sphere.header.stamp = stamp
    s_sphere.ns = 'pt'
    s_sphere.id = 0
    s_sphere.type = Marker.SPHERE
    s_sphere.action = Marker.ADD
    s_sphere.pose.position = Point(x=sx, y=sy, z=sz)
    s_sphere.pose.orientation.w = 1.0
    s_sphere.scale = Vector3(x=0.04, y=0.04, z=0.04)
    s_sphere.color = ColorRGBA(r=1.0, g=0.8, b=0.0, a=1.0)
    s_text = Marker()
    s_text.header.frame_id = REFERENCE_FRAME
    s_text.header.stamp = stamp
    s_text.ns = 'txt'
    s_text.id = 0
    s_text.type = Marker.TEXT_VIEW_FACING
    s_text.action = Marker.ADD
    s_text.pose.position = Point(x=sx, y=sy, z=sz + 0.08)
    s_text.pose.orientation.w = 1.0
    s_text.scale.z = 0.04
    s_text.color = COLOR_TEXT
    s_text.text = 'Start'
    new_markers.extend([s_sphere, s_text])

    for i, tgt in enumerate(targets):
        tx, ty, tz = tgt['pos']
        c = COLORS[i % len(COLORS)]
        sphere = Marker()
        sphere.header.frame_id = REFERENCE_FRAME
        sphere.header.stamp = stamp
        sphere.ns = 'pt'
        sphere.id = i + 1
        sphere.type = Marker.SPHERE
        sphere.action = Marker.ADD
        sphere.pose.position = Point(x=tx, y=ty, z=tz)
        sphere.pose.orientation.w = 1.0
        sphere.scale = Vector3(x=0.04, y=0.04, z=0.04)
        sphere.color = c
        text = Marker()
        text.header.frame_id = REFERENCE_FRAME
        text.header.stamp = stamp
        text.ns = 'txt'
        text.id = i + 1
        text.type = Marker.TEXT_VIEW_FACING
        text.action = Marker.ADD
        text.pose.position = Point(x=tx, y=ty, z=tz + 0.08)
        text.pose.orientation.w = 1.0
        text.scale.z = 0.04
        text.color = COLOR_TEXT
        text.text = tgt['label']
        # 허용 오차 링 — tol_rad 를 시각적 반경으로 환산
        ring = Marker()
        ring.header.frame_id = REFERENCE_FRAME
        ring.header.stamp = stamp
        ring.ns = 'ring'
        ring.id = i
        ring.type = Marker.CYLINDER
        ring.action = Marker.ADD
        ring.pose.position = Point(x=tx, y=ty, z=tz)
        ring.pose.orientation.w = 1.0
        ring.scale.x = tol_rad * 0.4
        ring.scale.y = tol_rad * 0.4
        ring.scale.z = 0.005
        ring.color = ColorRGBA(r=c.r, g=c.g, b=c.b, a=0.25)
        new_markers.extend([sphere, text, ring])

    _markers.markers = new_markers
    marker_pub.publish(_markers)

## 8. 시나리오 — 그리퍼 아래 방향 유지 + 3 지점 순회

시작점은 base 로부터 ~45cm 전방, 55cm 높이.
타깃 A/B/C 는 시작점 주변 좌우/위쪽에 배치 — FR3 워크스페이스 안에서 잘 잡힌다.

In [ ]:
# 1. ready 로 초기화
import time
node.get_logger().info('--- ready 자세로 초기 이동 ---')
go_to_joint_goal(ready_target)
time.sleep(1.0)

# 2. 시작점 / 타깃 / 제약 정의
start_pos = (0.45, 0.00, 0.55)
start_pose = make_pose(*start_pos, math.pi, 0.0, 0.0)

down_quaternion = euler_to_quaternion(math.pi, 0.0, 0.0)
TOL = 0.3   # ±17° 정도

targets = [
    {'label': 'A',  'pos': (0.45, -0.15, 0.45)},
    {'label': 'B',  'pos': (0.45,  0.15, 0.45)},
    {'label': 'C',  'pos': (0.40,  0.00, 0.65)},
]

publish_targets(start_pos, targets, TOL)
node.get_logger().info(
    f'방향 제약: 그리퍼 아래(roll=π) 유지, 허용 ±{math.degrees(TOL):.0f}° (Z축 자유)'
)

## 9. 시작 위치로 (제약 없이) 이동

In [ ]:
node.get_logger().info('--- 시작 위치로 이동 (제약 없이) ---')
go_to_pose_goal(start_pose)
time.sleep(1.0)

## 10. 타깃 A 로 — 경로 제약 하 이동

In [ ]:
tgt = targets[0]
node.get_logger().info(f"--- {tgt['label']} 로 제약 이동 ---")
ok = go_to_pose_with_path_constraint(
    make_pose(*tgt['pos'], math.pi, 0.0, 0.0),
    path_orientation=down_quaternion,
    tol_xy=TOL, tol_z=3.14,
)
node.get_logger().info(f"  결과: {'성공' if ok else '실패'}")
time.sleep(1.0)

# 시작 위치로 복귀 (제약 하)
go_to_pose_with_path_constraint(start_pose, down_quaternion, TOL)
time.sleep(0.5)

## 11. 타깃 B 로 — 경로 제약 하 이동

In [ ]:
tgt = targets[1]
node.get_logger().info(f"--- {tgt['label']} 로 제약 이동 ---")
ok = go_to_pose_with_path_constraint(
    make_pose(*tgt['pos'], math.pi, 0.0, 0.0),
    path_orientation=down_quaternion,
    tol_xy=TOL, tol_z=3.14,
)
node.get_logger().info(f"  결과: {'성공' if ok else '실패'}")
time.sleep(1.0)

go_to_pose_with_path_constraint(start_pose, down_quaternion, TOL)
time.sleep(0.5)

## 12. 타깃 C 로 — 경로 제약 하 이동

In [ ]:
tgt = targets[2]
node.get_logger().info(f"--- {tgt['label']} 로 제약 이동 ---")
ok = go_to_pose_with_path_constraint(
    make_pose(*tgt['pos'], math.pi, 0.0, 0.0),
    path_orientation=down_quaternion,
    tol_xy=TOL, tol_z=3.14,
)
node.get_logger().info(f"  결과: {'성공' if ok else '실패'}")
time.sleep(1.0)

go_to_pose_with_path_constraint(start_pose, down_quaternion, TOL)
time.sleep(0.5)

## 13. 비교 — 같은 타깃 B 를 **제약 없이** 이동

위에서는 path_constraints 가 적용된 plan 을 보냈고, 여기서는 일반 `go_to_pose_goal` 만 부른다.
RViz 에서 trajectory 의 끝단 방향이 어떻게 달라지는지 비교해 보자 (제약 없으면 손목이 휙 회전할 수 있음).

In [ ]:
node.get_logger().info('--- 제약 없이 B 로 이동 (비교) ---')
go_to_pose_goal(make_pose(*targets[1]['pos'], math.pi, 0.0, 0.0))
time.sleep(1.0)
go_to_pose_goal(start_pose)
time.sleep(0.5)

## 14. ready 복귀

In [ ]:
node.get_logger().info('--- ready 복귀 ---')
go_to_joint_goal(ready_target)
node.get_logger().info('=== franka_ex08 완료! ===')

## 15. 정리

In [ ]:
node.destroy_node()
try:
    rclpy.shutdown()
except Exception:
    pass